# Segment Revenue Extraction
从 CNINFO 年报 PDF 提取分部营收，写入 `data.json`。

**公司 & 分部**：
- 三安光电 600703 → 集成电路产品
- 士兰微   600460 → 分立器件产品
- 华润微   688396 → 产品与方案

**运行顺序**：Cell 1 → 2 → 3 → **4（下载 PDF）** → **5（提取 dry-run）** → **6（写入 & push）**

In [ ]:
# Cell 1 — 安装依赖（Colab 已 clone 仓库时 cwd 自动在 analog-pd-dashboard）
!pip install -q requests google-cloud-storage google-cloud-secret-manager google-genai
import os, subprocess
# 确保在仓库目录
if os.path.exists('/content/analog-pd-dashboard'):
    os.chdir('/content/analog-pd-dashboard')
print('Working dir:', os.getcwd())
!git checkout claude/code-review-UrrfV
!git pull origin claude/code-review-UrrfV

In [ ]:
# Cell 2 — GCP 认证（GCS 上传需要）
from google.colab import auth
auth.authenticate_user()
print('GCP authenticated ✓')

In [ ]:
# Cell 3 — 设置 Cookie & Gemini Key
from getpass import getpass

os.environ['CNINFO_COOKIE'] = getpass('Paste CNINFO_COOKIE: ')

r = subprocess.run(
    ['gcloud', 'secrets', 'versions', 'access', 'latest',
     '--secret=VITE_GEMINI_API_KEY', '--project=st-china-ai-force'],
    capture_output=True, text=True
)
os.environ['GEMINI_API_KEY'] = r.stdout.strip()
print('CNINFO_COOKIE set ✓')
print('GEMINI_API_KEY set:', bool(os.environ.get('GEMINI_API_KEY')))

In [ ]:
# Cell 4 — 下载所有年份 PDF 到 GCS（仅下载，不调用 Gemini）
# 三安已有缓存，跳过；士兰微和华润微首次下载
!python fetch_segment_rev_pdf.py --download-only --companies Silan 'CR Micro' --years 2019 2020 2021 2022 2023 2024 2025

In [ ]:
# Cell 5 — Gemini 提取 + dry-run 预览（不写入）
# 确认数字正确后再运行 Cell 6
!python fetch_segment_rev_pdf.py --companies Silan 'CR Micro' --dry-run

In [ ]:
# Cell 6 — 写入 data.json 并 push
# 只有 Cell 5 数字确认无误后才运行
!python fetch_segment_rev_pdf.py --companies Silan 'CR Micro'

!git config user.email 'jania.jiang@gmail.com'
!git config user.name 'Jania Jiang'
!git add data.json
!git commit -m 'data: Silan 分立器件产品 & CR Micro 产品与方案 segment revenue (PDF-extracted)'
!git push origin claude/code-review-UrrfV